# Bin Buddy · WS1 · Notebook 03 — Explainability: Grad-CAM and its alternatives

Every Bin Buddy verdict shows a heatmap of *which part of the photo drove the prediction*
(proposal objective O4). This notebook explains how that heatmap is computed, compares five
class-activation-map (CAM) methods, and **tests whether the maps can be trusted** — a heatmap
that looks plausible is not automatically a faithful explanation.

### How Grad-CAM works (Selvaraju et al., 2017)
A CNN's last convolutional layer produces *K* feature maps $A^k$ (e.g. 1280 maps of 7×7 for
EfficientNet-B0), each responding to some visual pattern. For class *c* with score $y^c$:

1. **Importance of each map**: $\alpha_k^c = \frac{1}{Z}\sum_{i,j} \frac{\partial y^c}{\partial A^k_{ij}}$
   — the average gradient: how much the class score rises if map *k* gets stronger.
2. **Heatmap**: $L^c = \mathrm{ReLU}\left(\sum_k \alpha_k^c A^k\right)$, upsampled to the image.
   ReLU keeps only regions that *increase* the class score.

### The five methods compared
| Method | Weights $\alpha_k$ from | Class-specific? | Cost |
|---|---|---|---|
| **Grad-CAM** | average gradient | yes | 1 forward + 1 backward |
| **Grad-CAM++** | weighted (2nd/3rd-order) gradients, better for several objects | yes | 1 + 1 |
| **Score-CAM** | no gradients: score increase when the image is masked by each map | yes | K forwards (slow) |
| **LayerCAM** | element-wise positive gradients (finer detail) | yes | 1 + 1 |
| **EigenCAM** | first principal component of the feature maps | **no** | 1 forward |

### How we judge them
- **Deletion AUC** (lower is better): remove (blur) the most important pixels first; a faithful
  map makes the class probability collapse quickly.
- **Insertion AUC** (higher is better): start from a blurred image and restore the most important
  pixels first; the probability should recover quickly.
- **Sanity check** (Adebayo et al., 2018): randomise the classifier weights. A method that truly
  depends on what the model learned must produce a *different* map (low rank correlation).

## 0. Setup

Run on **Google Colab** (Runtime → Change runtime type → **T4 GPU** for notebooks 02–04; CPU is
enough for 01). The cell below mounts Google Drive, clones the team repository (branch
`ws1/vision-model`) and installs pinned packages, so every run uses the same code and versions.

`SMOKE=1` is only for the developer's local test run on tiny synthetic data; leave it unset on Colab.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

SMOKE = os.environ.get("BINBUDDY_SMOKE") == "1"  # local test on synthetic data only
REPO_URL = "https://github.com/DevelopingGod/Group-5-PRS-Practice-Module-2026.git"
BRANCH = "ws1/vision-model"
PINNED = ["timm==1.0.30", "grad-cam==1.5.7", "open-clip-torch==3.3.0", "imagehash==4.3.2"]

if SMOKE:
    REPO = Path(os.environ["BINBUDDY_REPO"])
    DATA = Path(os.environ["BINBUDDY_DATA_ROOT"])
    WORK = DATA / "_work"
else:
    from google.colab import drive

    drive.mount("/content/drive")
    REPO = Path("/content/repo")
    DATA = Path("/content/drive/MyDrive/BinBuddy_Data")
    WORK = Path("/content/work")
    if not REPO.exists():
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "-q"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINNED], check=True)

# The contracts and helpers come straight from the repository (no copy-paste drift).
for p in (REPO / "contracts", REPO / "ml", REPO / "services" / "inference"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
WORK.mkdir(parents=True, exist_ok=True)
PREP = WORK / "prepared"  # resized images, extracted from DATA / "prepared_v1.tar"
MANIFEST = DATA / "manifests" / "vision_manifest.csv"
RESULTS = DATA / "results"
MODELS_DIR = DATA / "models"
for d in (MANIFEST.parent, RESULTS, MODELS_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("repo:", REPO, "| data:", DATA, "| smoke:", SMOKE)

In [ ]:
import tarfile

import pandas as pd

TAR = DATA / "prepared_v1.tar"
if not PREP.exists():
    with tarfile.open(TAR) as tf:  # one big file from Drive is much faster than 10k small ones
        tf.extractall(PREP, filter="data")
df = pd.read_csv(MANIFEST)
print(len(df), "images")
print(df.groupby(["source", "split"]).size().unstack(fill_value=0))

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import timm
import torch
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms as T

from binbuddy_contracts.enums import Material
from binbuddy_ml import experiments as ex

CLASSES = [m.value for m in Material]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PRETRAINED = os.environ.get("BINBUDDY_SMOKE_PRETRAINED", "1") == "1"
df["label"] = df["material"].map(CLASSES.index)
chosen = json.loads((RESULTS / "chosen_model.json").read_text())
EXP = chosen["experiment"]
FINE_TUNED = [m.key for m in ex.MODELS if m.mode is ex.Mode.FINE_TUNE]
# Grad-CAM needs a network trained end to end, so a probe/zero-shot winner is explained
# through the best fine-tuned model instead (and the report says so).
res = (
    pd.DataFrame(json.loads((RESULTS / "comparison_results.json").read_text()))
    if (RESULTS / "comparison_results.json").exists()
    else None
)
if chosen["model"] in FINE_TUNED:
    KEY = chosen["model"]
elif res is not None:
    ft = res[(res["experiment"] == EXP) & (res["mode"] == "fine_tune")]
    KEY = ft.sort_values("macro_f1", ascending=False).iloc[0]["model"]
else:
    KEY = "efficientnet_b0"
print("explaining:", KEY, "| experiment:", EXP)


def load(key):
    spec = ex.model(key)
    m = timm.create_model(spec.timm_id, pretrained=False, num_classes=len(CLASSES))
    weights = MODELS_DIR / f"{EXP}_{key}.pt"
    if weights.exists():
        m.load_state_dict(torch.load(weights, map_location="cpu"))
    else:
        print("WARNING: no fine-tuned weights for", key, "- using untrained head")
    return m.to(DEVICE).eval()


def eval_transform(model):
    cfg = timm.data.resolve_data_config({}, model=model)
    tf = T.Compose(
        [T.Resize(256), T.CenterCrop(224), T.ToTensor(), T.Normalize(cfg["mean"], cfg["std"])]
    )
    show = T.Compose([T.Resize(256), T.CenterCrop(224)])
    return tf, show


def vit_tokens_to_map(t):
    # DeiT: drop the class token, lay the 196 patch tokens back out as a 14 x 14 grid.
    t = t[:, 1:, :]
    side = int(t.shape[1] ** 0.5)
    return t.reshape(t.shape[0], side, side, t.shape[2]).permute(0, 3, 1, 2)


def target_layer(key, m):
    """Last spatial feature map before pooling, per architecture."""
    if key == "efficientnet_b0":
        return m.bn2, None  # conv_head -> BN -> SiLU output (1280 x 7 x 7)
    if key == "mobilenetv3_l":
        return m.blocks[-1], None  # MobileNetV3 pools *before* its conv_head
    if key == "resnet50":
        return m.layer4[-1], None
    if key == "convnext_t":
        return m.stages[-1], None
    if key == "swin_t":
        return m.layers[-1], lambda t: t.permute(0, 3, 1, 2)  # timm Swin is channels-last
    if key == "deit_s":
        return m.blocks[-1].norm1, vit_tokens_to_map
    raise KeyError(key)


test = df[(df["split"] == "test") & (df["source"] == "local")]
if not len(test):
    test = df[df["split"] == "test"]
N_EVAL = 4 if SMOKE else 40
eval_rows = test.sample(min(N_EVAL, len(test)), random_state=0)

In [ ]:
from pytorch_grad_cam import EigenCAM, GradCAM, GradCAMPlusPlus, LayerCAM, ScoreCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from scipy.stats import spearmanr

METHODS = {
    "Grad-CAM": GradCAM,
    "Grad-CAM++": GradCAMPlusPlus,
    "Score-CAM": ScoreCAM,
    "LayerCAM": LayerCAM,
    "EigenCAM": EigenCAM,
}
model = load(KEY)
tf, show = eval_transform(model)
layer, reshape = target_layer(KEY, model)


def cams(method_cls, m, lyr, rs, x, cls):
    with method_cls(model=m, target_layers=[lyr], reshape_transform=rs) as cam:
        if method_cls is ScoreCAM:
            cam.batch_size = 8 if SMOKE else 64
        return cam(input_tensor=x, targets=[ClassifierOutputTarget(int(c)) for c in cls])


def batch(rows):
    imgs = [Image.open(PREP / p).convert("RGB") for p in rows["path"]]
    x = torch.stack([tf(i) for i in imgs]).to(DEVICE)
    shown = [np.asarray(show(i), dtype=np.float32) / 255 for i in imgs]
    return x, shown

## 1. What the five methods show on the same photos

In [ ]:
grid_rows = eval_rows.head(5)
x, shown = batch(grid_rows)
with torch.no_grad():
    pred = model(x).softmax(1).cpu().numpy()
cls = pred.argmax(1)
maps = {name: cams(m, model, layer, reshape, x, cls) for name, m in METHODS.items()}
fig, axes = plt.subplots(
    len(grid_rows),
    1 + len(METHODS),
    figsize=(2.2 * (1 + len(METHODS)), 2.3 * len(grid_rows)),
    squeeze=False,
)
for r in range(len(grid_rows)):
    axes[r][0].imshow(shown[r])
    axes[r][0].set_title(
        f"true {grid_rows.iloc[r]['material']}\npred {CLASSES[cls[r]]} {pred[r].max():.0%}",
        fontsize=7,
    )
    for c, name in enumerate(METHODS, start=1):
        axes[r][c].imshow(show_cam_on_image(shown[r], maps[name][r], use_rgb=True))
        axes[r][c].set_title(name, fontsize=7)
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 2. Faithfulness: deletion and insertion curves

In [ ]:
STEPS = np.linspace(0, 1, 11)


def blurred(x):
    return T.functional.gaussian_blur(x, kernel_size=31, sigma=10.0)


@torch.no_grad()
def curve(x, heat, c, mode):
    """Probability of class c as the top-ranked pixels are removed (deletion) or restored (insertion)."""
    base = blurred(x)
    order = torch.from_numpy(heat.reshape(-1)).argsort(descending=True).to(x.device)
    out = []
    for frac in STEPS:
        mask = torch.zeros(heat.size, device=x.device)
        mask[order[: int(frac * heat.size)]] = 1
        mask = mask.reshape(1, 1, *heat.shape)
        img = x * (1 - mask) + base * mask if mode == "deletion" else base * (1 - mask) + x * mask
        out.append(model(img).softmax(1)[0, c].item())
    return np.array(out)


rows = []
for i in range(len(eval_rows)):
    xi, _ = batch(eval_rows.iloc[[i]])
    with torch.no_grad():
        c = int(model(xi).argmax(1))
    for name, m in METHODS.items():
        heat = cams(m, model, layer, reshape, xi, [c])[0]
        rows.append(
            {
                "method": name,
                "deletion_auc": float(np.trapezoid(curve(xi, heat, c, "deletion"), STEPS)),
                "insertion_auc": float(np.trapezoid(curve(xi, heat, c, "insertion"), STEPS)),
            }
        )
faith = pd.DataFrame(rows).groupby("method").agg(["mean", "std"]).round(3)
faith.columns = [f"{a} ({b})" for a, b in faith.columns]
faith

## 3. Sanity check: does the map depend on what the model learned?
The classifier layer is replaced by random weights. Faithful, class-specific methods should now
produce *different* maps (low Spearman correlation with the original). **EigenCAM is expected to
fail this test**: it only looks at the feature maps, never at the classifier — a useful lesson
on why a pretty map is not proof of an explanation.

In [ ]:
import copy

rand = copy.deepcopy(model)
torch.manual_seed(0)
for p in rand.get_classifier().parameters():
    torch.nn.init.normal_(p, std=0.02)
r_layer, _ = target_layer(KEY, rand)
xs, _ = batch(eval_rows.head(10))
with torch.no_grad():
    cs = model(xs).argmax(1).cpu().numpy()
sanity = []
for name, m in METHODS.items():
    a = cams(m, model, layer, reshape, xs, cs)
    b = cams(m, rand, r_layer, reshape, xs, cs)
    rho = [spearmanr(a[i].ravel(), b[i].ravel()).statistic for i in range(len(cs))]
    sanity.append({"method": name, "spearman_vs_randomised": float(np.nanmean(rho))})
sanity = pd.DataFrame(sanity).set_index("method").round(3)
faith.join(sanity)

## 4. Same method, different architectures
Grad-CAM on every fine-tuned model for the same photos. Transformers need their tokens laid back
out as a grid first (`reshape_transform`), and their coarser 14×14 / 7×7 maps look different from
CNN maps — part of the reason a CNN is preferred for the explained verdict.

In [ ]:
keys = [k for k in FINE_TUNED if (MODELS_DIR / f"{EXP}_{k}.pt").exists()] or [KEY]
few = eval_rows.head(3)
fig, axes = plt.subplots(
    len(few), len(keys), figsize=(2.2 * len(keys), 2.3 * len(few)), squeeze=False
)
for c, k in enumerate(keys):
    m = load(k)
    tf_k, _ = eval_transform(m)
    lyr, rs = target_layer(k, m)
    x = torch.stack([tf_k(Image.open(PREP / p).convert("RGB")) for p in few["path"]]).to(DEVICE)
    with torch.no_grad():
        cls_k = m(x).argmax(1).cpu().numpy()
    heat = cams(GradCAM, m, lyr, rs, x, cls_k)
    _, shown = batch(few)
    for r in range(len(few)):
        axes[r][c].imshow(show_cam_on_image(shown[r], heat[r], use_rgb=True))
        axes[r][c].set_title(f"{k}\n{CLASSES[cls_k[r]]}", fontsize=7)
        axes[r][c].axis("off")
    del m
plt.tight_layout()
plt.show()

## 5. Serving Grad-CAM without autograd (why the app can stay light)
If the classifier is *linear on the global-average-pooled* feature maps
($y^c = \sum_k w^c_k \cdot \frac{1}{Z}\sum_{ij} A^k_{ij} + b^c$), then
$\partial y^c / \partial A^k_{ij} = w^c_k / Z$ for every pixel, so $\alpha^c_k = w^c_k / Z$ and

$$L^c_{\text{Grad-CAM}} = \mathrm{ReLU}\Big(\sum_k w^c_k A^k\Big) / Z = L^c_{\text{CAM}} / Z.$$

After normalising to [0, 1] the two maps are **identical**. The deployed API can therefore compute
the heatmap from the exported feature map and classifier weights with plain NumPy — no PyTorch,
no backward pass. This holds exactly for EfficientNet-B0 and ResNet-50 (pool → linear), not for
MobileNetV3 (pool → 1×1 conv → activation → linear) or ConvNeXt (pool → LayerNorm → linear).
The check below measures it numerically.

In [ ]:
def cam_from_weights(m, x, c):
    feats = m.forward_features(x)  # B x K x h x w (CNNs)
    w = m.get_classifier().weight[c]  # K
    heat = F.relu(torch.einsum("k,bkhw->bhw", w, feats))
    heat = F.interpolate(heat[:, None], size=x.shape[-2:], mode="bilinear", align_corners=False)[
        :, 0
    ]
    heat = heat - heat.amin((1, 2), keepdim=True)
    return (heat / heat.amax((1, 2), keepdim=True).clamp_min(1e-8)).detach().cpu().numpy()


if KEY in {"efficientnet_b0", "resnet50"}:
    xs, _ = batch(eval_rows.head(8))
    with torch.no_grad():
        cs = model(xs).argmax(1).cpu().numpy()
    g = cams(GradCAM, model, layer, reshape, xs, cs)
    cw = np.stack([cam_from_weights(model, xs[i : i + 1], int(cs[i]))[0] for i in range(len(cs))])
    corr = [np.corrcoef(g[i].ravel(), cw[i].ravel())[0, 1] for i in range(len(cs))]
    print(
        f"Grad-CAM vs weight-CAM: mean Pearson r = {np.nanmean(corr):.4f}, max |diff| = {np.abs(g - cw).max():.4f}"
    )
else:
    print(KEY, "has a non-linear head: serve Grad-CAM with autograd (PyTorch) instead.")

## 6. Findings (fill in after running — for the report)
- Most faithful method on deletion / insertion: …
- Sanity check: methods that pass / fail (EigenCAM expected to fail): …
- CNN vs transformer maps: …
- **Method used in the app:** Grad-CAM, because … (and served as weight-CAM if the head is linear).
- Failure cases seen (map on background, on the hand, on the label rather than the material): …